# Sudoku-01 : Résolution par Backtracking (Rust)

**Navigation** : [<< Sudoku-00 Environment](Sudoku-00-Environment-CSharp.ipynb) | [Index](README.md) | [Jumeau Python](Sudoku-01-Backtracking-Python.ipynb) | [Sudoku-02 DancingLinks Python >>](Sudoku-02-DancingLinks-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Implémenter un algorithme de backtracking en Rust, bibliothèque standard seule, pour résoudre le Sudoku
2. Comprendre la complexité algorithmique du backtracking (O(9^m))
3. Appliquer l'heuristique MRV (Minimum Remaining Values) pour accélérer la recherche
4. Mesurer et comparer les performances sur des puzzles de difficultés variées

**Prérequis** : noyau `rust` (evcxr), notions d'algorithmes de recherche
**Durée estimée** : ~20 min

***

Ce notebook implémente un solveur de Sudoku utilisant l'algorithme de backtracking en Rust, sans aucune crate externe.
C'est l'équivalent Rust des carnets `Sudoku-01-Backtracking-Python.ipynb` et `Sudoku-01-Backtracking-CSharp.ipynb`.

In [2]:
// Imports
// Le noyau evcxr fournit la bibliotheque standard : aucune crate n'est declaree
// (lot « Rust std - Sudoku »). Le carnet Python importe numpy sans jamais
// l'utiliser ; il n'y a donc rien a remplacer de ce cote.
use std::path::{Path, PathBuf};
use std::time::Instant;

println!("Imports OK : std::fs, std::path, std::time (bibliotheque standard seule)");

Imports OK : std::fs, std::path, std::time (bibliotheque standard seule)


Configuration du chemin vers les fichiers de puzzles Sudoku.

In [3]:
// Configuration du chemin vers les puzzles

// Chercher le dossier Puzzles : le dossier du carnet, puis la racine du depot.
let here = std::env::current_dir().unwrap();
let mut racines: Vec<PathBuf> = vec![here.clone(), here.join("MyIA.AI.Notebooks").join("Sudoku")];
if let Some(parent) = here.parent() {
    racines.push(parent.to_path_buf().join("Sudoku"));
}

let mut puzzles_dir = PathBuf::from("Puzzles");
for racine in &racines {
    if racine.join("Puzzles").is_dir() {
        puzzles_dir = racine.join("Puzzles");
        break;
    }
}

// Verifier que le dossier existe
if puzzles_dir.is_dir() {
    println!("Dossier Puzzles: {}", puzzles_dir.file_name().unwrap_or_default().to_string_lossy());
} else {
    println!("ATTENTION: Dossier Puzzles non trouve a {}", puzzles_dir.display());
};

Dossier Puzzles: Puzzles


## 1. Structure SudokuGrid

La structure `Grid` encapsule la représentation d'une grille de Sudoku 9x9 et fournit des méthodes utilitaires.

### Représentation des données

- **Tableau 9x9** : `grid.cells[row][col]` où row et col vont de 0 à 8, typé `[[u8; 9]; 9]`
- **Cases vides** : Représentées par la valeur `0`
- **Valeurs valides** : 1 à 9 pour les cases remplies

### Méthodes principales

| Méthode | Description |
|---------|-------------|
| `from_string(s)` | Crée une grille depuis une chaîne de 81 caractères |
| `is_valid_placement(row, col, num)` | Vérifie si un placement respecte les contraintes |
| `find_empty()` | Trouve la première case vide (parcours ligne par ligne), rend `Option<(usize, usize)>` |
| `clone()` | Copie profonde, dérivée par `#[derive(Clone)]` sur un tableau de valeurs |
| `show()` | Rendu texte de la grille (l'équivalent de `__str__`) |

### Règles de validation

Un placement est valide si le nombre n'apparaît pas déjà dans :
1. **La même ligne** (9 cases horizontales)
2. **La même colonne** (9 cases verticales)
3. **Le même bloc 3x3** (l'un des 9 carrés de la grille)

In [4]:
// Structure Grid : la classe SudokuGrid du carnet Python
#[derive(Clone)]
struct Grid {
    cells: [[u8; 9]; 9],
}

impl Grid {
    fn new() -> Grid {
        Grid { cells: [[0u8; 9]; 9] }
    }

    fn from_string(s: &str) -> Grid {
        let s = s.replace('.', "0").replace(' ', "").replace('\n', "");
        assert!(s.len() == 81, "La chaine doit avoir 81 caracteres, recu {}", s.len());

        let mut grid = Grid::new();
        for (i, ch) in s.chars().enumerate() {
            grid.cells[i / 9][i % 9] = ch.to_digit(10).unwrap() as u8;
        }
        grid
    }

    fn is_valid_placement(&self, row: usize, col: usize, num: u8) -> bool {
        // Verifier la ligne
        if self.cells[row].contains(&num) {
            return false;
        }

        // Verifier la colonne
        for r in 0..9 {
            if self.cells[r][col] == num {
                return false;
            }
        }

        // Verifier le bloc 3x3
        let box_row = 3 * (row / 3);
        let box_col = 3 * (col / 3);
        for r in box_row..box_row + 3 {
            for c in box_col..box_col + 3 {
                if self.cells[r][c] == num {
                    return false;
                }
            }
        }

        true
    }

    fn find_empty(&self) -> Option<(usize, usize)> {
        for r in 0..9 {
            for c in 0..9 {
                if self.cells[r][c] == 0 {
                    return Some((r, c));
                }
            }
        }
        None
    }

    fn is_complete(&self) -> bool {
        !self.cells.iter().flatten().any(|&v| v == 0)
    }

    fn count_empty(&self) -> usize {
        self.cells.iter().flatten().filter(|&&v| v == 0).count()
    }

    fn to_string(&self) -> String {
        let mut s = String::with_capacity(81);
        for r in 0..9 {
            for c in 0..9 {
                s.push((b'0' + self.cells[r][c]) as char);
            }
        }
        s
    }

    fn show(&self) -> String {
        let mut lines: Vec<String> = Vec::new();
        for r in 0..9 {
            if r > 0 && r % 3 == 0 {
                lines.push("-".repeat(21));
            }
            let mut row_str = String::new();
            for c in 0..9 {
                if c > 0 && c % 3 == 0 {
                    row_str.push_str("| ");
                }
                let val = self.cells[r][c];
                row_str.push(if val != 0 { (b'0' + val) as char } else { '.' });
                row_str.push(' ');
            }
            lines.push(row_str);
        }
        lines.join("\n")
    }
}

// Test
let test_puzzle = "902005403100063025508407060026309001057010290090670530240530600705200304080041950".to_string();
let grid = Grid::from_string(&test_puzzle);
println!("Grille de test:");
println!("{}", grid.show());
println!("\nCases vides: {}", grid.count_empty());

Grille de test:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


. 2 6 | 3 . 9 | . . 1 


... (3 outputs left out)

2 4 . | 5 3 . | 6 . . 


7 . 5 | 2 . . | 3 . 4 


. 8 . | . 4 1 | 9 5 . 


Cases vides: 36


## 2. Solveur Backtracking Simple

L'implémentation récursive de l'algorithme de backtracking est élégante et concise.

### Pseudo-code

```
fonction backtrack(grille):
    case_vide = trouver_case_vide(grille)
    si case_vide est None:
        retourner True  # Solution trouvée!

    pour chaque valeur de 1 à 9:
        si placement_valide(case_vide, valeur):
            placer(case_vide, valeur)
            si backtrack(grille):
                retourner True
            retirer(case_vide)  # Backtrack

    retourner False  # Aucune solution avec cette configuration
```

En Rust, `backtrack` prend la grille en `&mut Grid` : placer un nombre et l'annuler sont deux affectations de case, sans allocation ni copie de la grille.

### Analyse de performance

Le compteur `call_count` permet de mesurer le travail effectué par l'algorithme. Plus le puzzle est difficile, plus il y aura d'appels récursifs car l'algorithme doit explorer plus de branches avant de trouver la solution.

### Optimisation potentielle

Le solveur simple parcourt les cases de gauche à droite, haut en bas. Ce n'est pas optimal car certaines cases ont moins de valeurs possibles que d'autres. L'heuristique **MRV** (présentée plus loin) améliore significativement les performances.

In [5]:
// Solveur Backtracking simple
struct BacktrackingSolver {
    call_count: usize,
}

impl BacktrackingSolver {
    fn new() -> BacktrackingSolver {
        BacktrackingSolver { call_count: 0 }
    }

    fn solve(&mut self, grid: &mut Grid) -> bool {
        self.call_count = 0;
        self.backtrack(grid)
    }

    fn backtrack(&mut self, grid: &mut Grid) -> bool {
        self.call_count += 1;

        // Trouver la prochaine case vide
        let (row, col) = match grid.find_empty() {
            Some(position) => position,
            None => return true, // Grille complete = solution trouvee
        };

        // Essayer les valeurs 1-9
        for num in 1..=9u8 {
            if grid.is_valid_placement(row, col, num) {
                // Placer le nombre
                grid.cells[row][col] = num;

                // Recurser
                if self.backtrack(grid) {
                    return true;
                }

                // Backtrack : annuler le placement
                grid.cells[row][col] = 0;
            }
        }

        false // Aucune valeur valide
    }
}

// Test du solveur
let mut solver = BacktrackingSolver::new();
let mut test_grid = Grid::from_string(&test_puzzle);

println!("Puzzle initial:");
println!("{}", test_grid.show());

let start = Instant::now();
let solved = solver.solve(&mut test_grid);
let elapsed = start.elapsed();

println!("\nResolu: {}", solved);
println!("Appels recursifs: {}", solver.call_count);
println!("Temps: {:.2} ms", elapsed.as_secs_f64() * 1000.0);
println!("\nSolution:");
println!("{}", test_grid.show());

Puzzle initial:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


. 2 6 | 3 . 9 | . . 1 


... (17 outputs left out)

---------------------


2 4 9 | 5 3 8 | 6 1 7 


7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


### Interprétation : Résultat du backtracking simple

Le solveur a résolu un puzzle facile avec seulement 49 appels récursifs en moins d'une milliseconde.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| **Appels récursifs** | 49 | Très peu d'explorations nécessaires |
| **Temps** | <1 ms | Résolution quasi-instantanée |
| **Cases vides initiales** | 36 | Puzzle relativement facile |

**Points clés** :
1. **Le puzzle est facile** : Beaucoup d'indices (45 cases remplies)
2. **Peu de backtracks** : L'algorithme fait les bons choix rapidement
3. **Performance acceptable** : Pour les puzzles faciles, le backtracking simple suffit

> **Note technique** : Sur ce puzzle, la première branche de l'arbre de recherche mène directement à la solution. C'est typique des puzzles faciles où les contraintes locales guident bien le solveur.

## Exercice : Vérifier qu'une grille est valide

### Énoncé

Après avoir résolu un Sudoku avec le backtracking, il est essentiel de vérifier que la solution obtenue est bien valide. Implémentez une fonction `is_valid_solution(grid)` qui vérifie **toutes** les contraintes du Sudoku sur une grille censée être complète.

### Ce que la fonction doit vérifier

1. **Pas de cases vides** : Aucune cellule ne contient 0
2. **Lignes valides** : Chaque ligne contient exactement les chiffres 1 à 9 (pas de doublon)
3. **Colonnes valides** : Chaque colonne contient exactement les chiffres 1 à 9
4. **Blocs 3x3 valides** : Chacun des 9 blocs contient exactement les chiffres 1 à 9

> **Indices :**

- Le tableau de référence est `(1..=9u8).collect::<Vec<u8>>()` : triez une copie de chaque ligne, colonne ou bloc et comparez-la
- Pour accéder au bloc 3x3 : `let (box_row, box_col) = (3 * (i / 3), 3 * (j / 3));`
- La fonction rend `Some(true)` si **toutes** les contraintes sont respectées, `Some(false)` sinon ; le stub rend `None` tant qu'elle n'est pas complétée

In [6]:
// EXERCICE : Verifier qu'une grille est valide

#[allow(unused_variables)]
fn is_valid_solution(grid: &Grid) -> Option<bool> {
    // TODO etudiant : implementez la verification
    // Etape 1 : verifier qu'il n'y a pas de cases vides (0)
    // Etape 2 : verifier que chaque ligne contient 1-9 sans doublon
    // Etape 3 : verifier que chaque colonne contient 1-9 sans doublon
    // Etape 4 : verifier que chaque bloc 3x3 contient 1-9 sans doublon
    None // TODO etudiant : remplacer par l'implementation
}

// Test : la solution du puzzle de test doit etre valide
let mut grid_check = Grid::from_string(&test_puzzle);
let mut solver_check = BacktrackingSolver::new();
solver_check.solve(&mut grid_check);
println!("Solution valide (puzzle de test) : {:?}", is_valid_solution(&grid_check)); // Some(true) attendu

// Test : le puzzle non resolu ne doit PAS etre valide
let grid_unsolved = Grid::from_string(&test_puzzle);
println!("Grille incomplete valide : {:?}", is_valid_solution(&grid_unsolved)); // Some(false) attendu

Solution valide (puzzle de test) : None


Grille incomplete valide : None


## 3. Chargement des puzzles depuis fichier

Les puzzles Sudoku sont stockés dans des fichiers texte, un puzzle par ligne (81 caractères représentant la grille ligne par ligne).

### Format des fichiers

- **81 caractères par ligne** : Chaque caractère représente une case (0-9)
- **0 ou .** : Case vide à résoudre
- **1-9** : Valeur initiale fixée

### Collections de puzzles disponibles

| Fichier | Difficulté | Nombre | Description |
|---------|------------|--------|-------------|
| `Sudoku_Easy51.txt` | Facile | 51 | Puzzles standards avec beaucoup d'indices |
| `Sudoku_hardest.txt` | Extreme | 11 | Top 11 des puzzles les plus difficiles connus |

Les puzzles "hardest" sont célèbres dans la communauté Sudoku car ils maximisent le nombre de backtracks nécessaires avec des algorithmes simples.

In [7]:
// Chargement des puzzles depuis un fichier
fn load_puzzles(filepath: &Path, max_puzzles: Option<usize>) -> Vec<String> {
    let content = std::fs::read_to_string(filepath).unwrap();
    let mut puzzles: Vec<String> = Vec::new();
    for line in content.lines() {
        let line = line.trim();
        if line.len() >= 81 {
            puzzles.push(line[..81].to_string());
            if let Some(max) = max_puzzles {
                if puzzles.len() >= max {
                    break;
                }
            }
        }
    }
    puzzles
}

// Charger les puzzles faciles
let easy_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), Some(10));
println!("Puzzles faciles charges: {}", easy_puzzles.len());

// Charger les puzzles difficiles
let hard_puzzles = load_puzzles(&puzzles_dir.join("Sudoku_hardest.txt"), None);
println!("Puzzles difficiles charges: {}", hard_puzzles.len());

Puzzles faciles charges: 10


Puzzles difficiles charges: 11


### Interprétation : Chargement des puzzles

Les fichiers de puzzles ont été chargés avec succès.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| **Puzzles faciles** | 10/51 | Echantillon représentatif de puzzles faciles |
| **Puzzles difficiles** | 11/11 | Collection complète des puzzles les plus durs |
| **Fichiers disponibles** | 6 collections | Difficultés croissantes pour benchmarks |

**Points clés** :
1. **Echantillonnage** : On charge 10 puzzles faciles pour un benchmark rapide
2. **Collection complète** : Les 11 puzzles "hardest" sont tous chargés pour tester les limites
3. **Format valide** : Tous les fichiers respectent le format 81 caractères par ligne

**Stratégie de benchmark** :
- Les puzzles faciles (10) permettent de vérifier la correction de base
- Les puzzles difficiles (11) testent les performances et la robustesse
- Les autres collections (top95, puzzles générés) ne sont pas utilisées ici mais restent disponibles

> **Note technique** : La fonction `load_puzzles` prend un paramètre `max_puzzles` en `Option<usize>` — `None` charge le fichier entier, là où le carnet Python utilise une valeur par défaut. Cela évite de charger inutilement les 51 puzzles faciles quand on veut seulement faire un test rapide. Le format du fichier est robuste : il accepte les caractères '0' ou '.' pour les cases vides.

## 4. Benchmark sur plusieurs puzzles

Le benchmark permet de comparer objectivement les performances du solveur sur différents niveaux de difficulté.

### Métriques mesurées

- **Taux de résolution** : Pourcentage de puzzles résolus (devrait être 100% pour un solveur correct)
- **Temps total/moyen** : Performance temporelle
- **Appels récursifs** : Mesure de la complexité effective (indépendante de la machine)

### Interprétation des résultats

- **Puzzles faciles** : Peu d'appels car beaucoup de cases sont déjà remplies et les contraintes guident rapidement vers la solution
- **Puzzles difficiles** : Beaucoup plus d'appels car l'algorithme doit explorer de nombreuses branches infructueuses avant de trouver le bon chemin

La différence entre puzzles faciles et difficiles peut être de plusieurs ordres de grandeur (10x à 1000x plus d'appels).

In [8]:
// Benchmark sur plusieurs puzzles
fn benchmark_solver(solver: &mut BacktrackingSolver, puzzles: &[String], name: &str) {
    println!("\n=== Benchmark: {} ({} puzzles) ===", name, puzzles.len());

    let mut total_time = 0.0f64;
    let mut total_calls = 0usize;
    let mut solved_count = 0usize;

    for (i, puzzle_str) in puzzles.iter().enumerate() {
        let mut grid = Grid::from_string(puzzle_str);
        let empty_count = grid.count_empty();

        let start = Instant::now();
        let solved = solver.solve(&mut grid);
        let elapsed = start.elapsed().as_secs_f64();

        total_time += elapsed;
        total_calls += solver.call_count;
        if solved {
            solved_count += 1;
        }

        // Afficher les premiers et les echecs
        if i < 5 || !solved {
            let status = if solved { "OK" } else { "ECHEC" };
            println!("  Puzzle {}: {}, {} vides, {} appels, {:.2} ms",
                     i + 1, status, empty_count, solver.call_count, elapsed * 1000.0);
        }
    }

    println!("\nResume:");
    println!("  Resolus: {}/{}", solved_count, puzzles.len());
    println!("  Temps total: {:.2} ms", total_time * 1000.0);
    println!("  Temps moyen: {:.2} ms", (total_time / puzzles.len() as f64) * 1000.0);
    println!("  Appels totaux: {}", total_calls);
    println!("  Appels moyens: {}", total_calls / puzzles.len());
}

// Benchmark
let mut solver = BacktrackingSolver::new();
benchmark_solver(&mut solver, &easy_puzzles, "Puzzles Faciles");
benchmark_solver(&mut solver, &hard_puzzles, "Puzzles Difficiles");

=== Benchmark: Puzzles Faciles (10 puzzles) ===


  Puzzle 1: OK, 36 vides, 49 appels, 0.00 ms


  Puzzle 2: OK, 49 vides, 201 appels, 0.03 ms


  Puzzle 3: OK, 51 vides, 295 appels, 0.02 ms


  Puzzle 4: OK, 53 vides, 19023 appels, 0.69 ms


  Puzzle 5: OK, 51 vides, 1683 appels, 0.06 ms


... (14 outputs left out)

  Temps total: 50.24 ms


  Temps moyen: 4.57 ms


  Appels totaux: 1050007


  Appels moyens: 95455


### Interprétation : Benchmark du backtracking simple

Les résultats montrent une différence énorme entre les puzzles faciles et difficiles.

| Type | Appels moyens | Analyse |
|------|---------------|---------|
| **Faciles** | ~25,000 | Performance acceptable |
| **Difficiles** | ~95,000 | 4x plus d'appels récursifs |

**Observations clés** :
1. **Variabilité énorme** : Puzzle 4 facile nécessite 19,023 appels vs 49 pour puzzle 1
2. **Explosion combinatoire** : Quelques cases vides supplémentaires = explosion du nombre d'appels
3. **Temps proportionnel aux appels** : le temps de résolution croît avec le nombre d'appels (ms absolus, machine-dépendants : cf. sortie benchmark ci-dessus)

**Exemple de puzzle difficile** : Puzzle 1 avec 59 cases vides nécessite 335,638 appels.

> **Note technique** : Le backtracking simple explore l'arbre de recherche de gauche à droite, sans stratégie intelligente. Sur les puzzles difficiles, cela signifie explorer des milliers de branches infructueuses avant de trouver la solution.

## Exercice : Comparer les performances Backtracking simple vs MRV

**Objectif :**
Comparez les performances du solveur backtracking simple et du solveur MRV
sur les puzzles de différentes difficultés.

**Indice :**
Utilisez la fonction `benchmark_solver` déjà définie avec les deux solveurs.
Affichez les résultats dans un tableau comparatif. La fonction rend `Option<(usize, usize)>`
— les appels totaux du solveur simple, puis ceux du solveur MRV — et `None` tant qu'elle n'est
pas complétée.

In [9]:
// EXERCICE : Comparer les performances Backtracking simple vs MRV

#[allow(unused_variables)]
fn compare_solvers(puzzles: &[String]) -> Option<(usize, usize)> {
    // TODO etudiant : comparez les deux solveurs sur les memes puzzles
    // Retournez les appels totaux de chaque solveur
    None // TODO etudiant
}

// Test : la comparaison renvoie None tant que l'exercice n'est pas complete
let resultat_comparaison = compare_solvers(&easy_puzzles[..3]);
println!("Resultat de la comparaison : {:?}", resultat_comparaison); // None attendu (a completer)

Resultat de la comparaison : None


## 5. Optimisation: Backtracking avec MRV (Minimum Remaining Values)

L'heuristique **MRV** (aussi appelée "Most Constrained Variable" ou "Fail-First") est l'une des améliorations les plus efficaces du backtracking.

### Principe

Au lieu de choisir la première case vide rencontrée, MRV sélectionne la case avec le **moins de valeurs possibles**. Cette stratégie :

1. **Détecte les échecs plus tôt** : Une case avec 0 valeurs possibles indique immédiatement une impasse
2. **Réduit le facteur de branchement** : Moins de choix = moins de branches à explorer
3. **Propage implicitement les contraintes** : Les cases les plus contraintes sont traitées en premier

### Exemple concret

```
Situation:
- Case A: 5 valeurs possibles {1,3,5,7,9}
- Case B: 2 valeurs possibles {4,6}

Sans MRV: On traite A d'abord -> 5 branches à explorer
Avec MRV: On traite B d'abord -> 2 branches seulement
```

> **Note Rust** : les valeurs possibles sont un `Vec<u8>` **trié croissant**, là où le carnet Python itère un `set` (dont l'ordre suit les emplacements de hachage). Cet ordre est celui des valeurs essayées, donc celui de l'arbre de recherche : l'égalité des compteurs MRV avec le carnet Python est **mesurée** sur ces onze puzzles, elle n'est pas une propriété de l'algorithme.

### Impact sur les performances

| Puzzle | Backtracking simple | Avec MRV | Speedup (appels) |
|--------|---------------------|----------|------------------|
| Facile | ~500 appels | ~200 appels | 2-3x |
| Difficile | ~500,000 appels | ~5,000 appels | souvent >100x |

> Ces ordres de grandeur sont indicatifs ; les valeurs réellement mesurées sur les puzzles de test figurent dans la section d'interprétation ci-dessous. La colonne « Speedup » y est un rapport du **nombre d'appels récursifs**, pas un rapport de temps : chaque appel MRV balaie les cases vides pour choisir la plus contrainte, il coûte donc plus cher qu'un appel simple.

L'amélioration est particulièrement spectaculaire sur les puzzles difficiles, où l'élagage précoce de l'arbre de recherche évite des centaines de milliers d'appels inutiles.

### Autres heuristiques (non implémentées ici)

- **Degree Heuristic** : Choisir la variable qui contraint le plus d'autres variables
- **Least Constraining Value** : Essayer d'abord les valeurs qui éliminent le moins de possibilités pour les voisins
- **Arc Consistency (AC-3)** : Propager les contraintes pour réduire les domaines


In [10]:
// Optimisation : backtracking avec MRV
struct MRVBacktrackingSolver {
    call_count: usize,
}

impl MRVBacktrackingSolver {
    fn new() -> MRVBacktrackingSolver {
        MRVBacktrackingSolver { call_count: 0 }
    }

    fn possible_values(&self, grid: &Grid, row: usize, col: usize) -> Vec<u8> {
        if grid.cells[row][col] != 0 {
            return Vec::new();
        }

        // Vec trie croissant : l'ordre des valeurs essayees est celui de l'arbre
        let mut possible: Vec<u8> = (1..=9u8).collect();

        // Retirer les valeurs de la ligne
        possible.retain(|&v| !grid.cells[row].contains(&v));

        // Retirer les valeurs de la colonne
        possible.retain(|&v| (0..9).all(|r| grid.cells[r][col] != v));

        // Retirer les valeurs du bloc
        let box_row = 3 * (row / 3);
        let box_col = 3 * (col / 3);
        possible.retain(|&v| {
            (box_row..box_row + 3).all(|r| (box_col..box_col + 3).all(|c| grid.cells[r][c] != v))
        });

        possible
    }

    fn find_mrv_empty(&self, grid: &Grid) -> Option<(usize, usize, Vec<u8>)> {
        let mut best: Option<(usize, usize, Vec<u8>)> = None;
        let mut best_count = 10usize;

        for r in 0..9 {
            for c in 0..9 {
                if grid.cells[r][c] == 0 {
                    let possible = self.possible_values(grid, r, c);
                    if possible.len() < best_count {
                        best_count = possible.len();
                        best = Some((r, c, possible));
                        if best_count == 0 {
                            return best; // Echec immediat
                        }
                    }
                }
            }
        }

        best
    }

    fn solve(&mut self, grid: &mut Grid) -> bool {
        self.call_count = 0;
        self.backtrack(grid)
    }

    fn backtrack(&mut self, grid: &mut Grid) -> bool {
        self.call_count += 1;

        let (row, col, possible) = match self.find_mrv_empty(grid) {
            Some(result) => result,
            None => return true, // Grille complete
        };

        if possible.is_empty() {
            return false; // Impasse
        }

        for num in possible {
            grid.cells[row][col] = num;
            if self.backtrack(grid) {
                return true;
            }
            grid.cells[row][col] = 0;
        }

        false
    }
}

// Comparaison
println!("=== Comparaison: Backtracking simple vs MRV ===");

let mut simple_solver = BacktrackingSolver::new();
let mut mrv_solver = MRVBacktrackingSolver::new();

for (i, puzzle_str) in hard_puzzles.iter().take(5).enumerate() {
    println!("\nPuzzle difficile {}:", i + 1);

    // Backtracking simple
    let mut grid1 = Grid::from_string(puzzle_str);
    let start = Instant::now();
    simple_solver.solve(&mut grid1);
    let t1 = start.elapsed().as_secs_f64() * 1000.0;

    // Backtracking MRV
    let mut grid2 = Grid::from_string(puzzle_str);
    let start = Instant::now();
    mrv_solver.solve(&mut grid2);
    let t2 = start.elapsed().as_secs_f64() * 1000.0;

    println!("  Simple: {} appels, {:.2} ms", simple_solver.call_count, t1);
    println!("  MRV:    {} appels, {:.2} ms", mrv_solver.call_count, t2);
    println!("  Speedup: {:.1}x", simple_solver.call_count as f64 / mrv_solver.call_count as f64);
};

=== Comparaison: Backtracking simple vs MRV ===


Puzzle difficile 1:


  Simple: 335638 appels, 9.27 ms


  MRV:    4037 appels, 11.06 ms


  Speedup: 83.1x


Puzzle difficile 2:


... (11 outputs left out)

Puzzle difficile 5:


  Simple: 207075 appels, 11.73 ms


  MRV:    79 appels, 0.20 ms


  Speedup: 2621.2x


### Interprétation : Impact de l'heuristique MRV

La comparaison montre que **MRV réduit massivement le nombre d'appels récursifs** du backtracking. Attention à la lecture : la colonne « Speedup » est le rapport des **nombres d'appels**, pas des temps.

| Puzzle | Simple appels | MRV appels | Speedup (appels) | Temps simple | Temps MRV |
|--------|---------------|------------|------------------|--------------|-----------|
| Puzzle 1 | 335,638 | 4,037 | **83.1x** | 9.27 ms | 11.06 ms |
| Puzzle 2 | 10,008 | 543 | **18.4x** | 0.34 ms | 1.36 ms |
| Puzzle 3 | 228,215 | 171 | **1,334.6x** | 10.22 ms | 0.43 ms |
| Puzzle 4 | 75,446 | 1,079 | **69.9x** | 2.67 ms | 2.85 ms |
| Puzzle 5 | 207,075 | 79 | **2,621.2x** | 11.73 ms | 0.20 ms |

**Points clés** :
1. **Moins d'appels, pas toujours moins de temps** : MRV explore jusqu'à 2 621 fois moins d'appels (puzzle 5), mais le temps mur ne baisse que sur deux puzzles sur cinq — puzzle 3 : 10.22 → 0.43 ms, puzzle 5 : 11.73 → 0.20 ms — et monte sur les trois autres (puzzle 1 : 9.27 → 11.06 ms). Chaque appel MRV balaie les cases vides pour choisir la plus contrainte : il coûte plus cher que l'appel simple.
2. **Variabilité réduite du nombre d'appels** : MRV stabilise l'exploration (79 à 4 037 appels, contre 10 008 à 335 638).
3. **Principe fail-first** : MRV détecte les impasses immédiatement, ce qui supprime des branches entières de l'arbre.

**Pourquoi MRV fonctionne si bien ?**
- Les cases avec peu de valeurs possibles sont les plus contraintes
- Les échecs sont détectés tôt, avant d'explorer des branches inutiles
- Sur Sudoku, les contraintes locales créent rapidement des "singletons"

> **Note technique** : MRV est une heuristique "fail-first" : elle privilégie les variables les plus susceptibles d'échouer, ce qui permet d'élaguer l'arbre de recherche dès le début. C'est particulièrement efficace sur Sudoku car les contraintes sont très locales.


## 6. Visualisation en texte

La représentation texte aide à comprendre la structure d'un Sudoku et à distinguer les valeurs initiales des valeurs trouvées par le solveur.

### Code couleur

Le carnet Python trace la grille avec matplotlib (noir pour les valeurs initiales, bleu pour les valeurs ajoutées). Aucune crate de tracé n'est autorisée dans ce lot : la bibliothèque standard ne sait pas dessiner, le rendu est donc textuel et la distinction passe par deux impressions et un décompte.

- **Cases `.`** : valeurs à trouver (données du puzzle)
- **Chiffres 1-9** : valeurs présentes dans la grille affichée

### Structure visuelle

La grille est divisée en 9 blocs 3x3 séparés par des lignes de tirets. Cette division est fondamentale pour les contraintes du Sudoku : chaque bloc doit contenir exactement une fois chaque chiffre de 1 à 9.

In [11]:
// Visualisation : rendu texte (bibliotheque standard seule)
fn render_pair(initial: &Grid, solved: &Grid) {
    println!("Puzzle initial :");
    println!("{}", initial.show());
    println!();
    println!("Solution (les '.' du puzzle initial ont ete remplis) :");
    println!("{}", solved.show());

    let mut vides = 0usize;
    let mut ajoutees = 0usize;
    for r in 0..9 {
        for c in 0..9 {
            if initial.cells[r][c] == 0 {
                vides += 1;
                if solved.cells[r][c] != 0 {
                    ajoutees += 1;
                }
            }
        }
    }
    println!("\nCases du puzzle : {}", 81 - vides);
    println!("Cases ajoutees par le solveur : {}", ajoutees);
}

// Exemple
let initial_grid = Grid::from_string(&easy_puzzles[0]);
let mut solved_grid = initial_grid.clone();
let mut solver = MRVBacktrackingSolver::new();
solver.solve(&mut solved_grid);

render_pair(&initial_grid, &solved_grid);

Puzzle initial :


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


. 2 6 | 3 . 9 | . . 1 


... (16 outputs left out)

7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


Cases du puzzle : 45


Cases ajoutees par le solveur : 36


### Interprétation : Visualisation de la solution

Le rendu texte permet de distinguer les valeurs initiales des valeurs trouvées par le solveur, et le décompte les chiffres.

| Aspect | Observation | Signification |
|--------|-------------|---------------|
| **Cases du puzzle** | 45 cases | Valeurs initiales du puzzle (indices) |
| **Cases ajoutées** | 36 cases | Valeurs déduites par le solveur MRV |
| **Structure 3x3** | 9 blocs visibles | Contraintes spatiales du Sudoku |

**Points clés** :
1. **Distribution uniforme** : Les cases ajoutées sont réparties sur toute la grille
2. **Blocs 3x3** : Chaque bloc contient exactement une fois les chiffres 1-9
3. **Vérification visuelle** : Le décompte initial + ajouté doit valoir 81

**Analyse de la résolution** :
- Le solveur MRV a rempli toutes les cases vides
- La solution respecte toutes les contraintes (lignes, colonnes, blocs)
- Le temps de résolution est imperceptible pour l'utilisateur

> **Note technique** : La fonction `render_pair` imprime la grille initiale et la grille résolue, puis compte les cases ajoutées. Le carnet Python fait le même travail avec `plot_sudoku`, qui accepte un paramètre optionnel `initial` pour colorer les valeurs ajoutées ; la couleur n'ayant pas d'équivalent en bibliothèque standard, c'est le décompte qui porte ici la même information.

## Navigation

### Notebooks Rust de cette série
- **Ce notebook** : Backtracking - Algorithme de base et heuristique MRV
- [Sudoku-04-SimulatedAnnealing-Rust](Sudoku-04-SimulatedAnnealing-Rust.ipynb) - Recuit simulé
- [Sudoku-07-Norvig-Rust](Sudoku-07-Norvig-Rust.ipynb) - Solveur par propagation de contraintes
- [Sudoku-14-BDD-Rust](Sudoku-14-BDD-Rust.ipynb) - Diagrammes de décision binaires

### Notebooks Python et C# équivalents
- [Sudoku-01-Backtracking-Python](Sudoku-01-Backtracking-Python.ipynb) - Backtracking en Python, jumeau de référence
- [Sudoku-01-Backtracking-CSharp](Sudoku-01-Backtracking-CSharp.ipynb) - Backtracking en C#
- [Sudoku-00-Environment-CSharp](Sudoku-00-Environment-CSharp.ipynb) - Configuration et benchmarks
- [Sudoku-02-DancingLinks-Python](Sudoku-02-DancingLinks-Python.ipynb) - Algorithm X / Dancing Links

## Exercice : Compter le nombre de solutions

### Principe

Un Sudoku bien formé a exactement une solution. La fonction `count_solutions(grid)` compte le nombre de solutions d'un puzzle en utilisant le backtracking avec l'heuristique MRV.

L'idée est de modifier l'algorithme de backtracking pour **ne pas s'arrêter** après la première solution, mais continuer à explorer toutes les branches possibles en incrémentant un compteur à chaque grille complète. Pour éviter les calculs infinis, on plafonne la recherche à `max_solutions` solutions.

### Points techniques

- **Copie de la grille** : On travaille sur une copie pour ne pas modifier l'original (`grid.clone()`)
- **MRV** : On réutilise `find_mrv_empty` pour choisir la case la plus contrainte
- **Arrêt anticipé** : Dès que `max_solutions` sont trouvées, on arrête la recherche

### Résultats attendus

- Un puzzle bien formé (1 seule solution) retourne `1`
- Un puzzle mal formé (trous importants) peut retourner `2` ou plus

In [12]:
// EXERCICE : Compter le nombre de solutions

#[allow(dead_code)]
fn get_possible_values(grid: &Grid, row: usize, col: usize) -> Vec<u8> {
    if grid.cells[row][col] != 0 {
        return Vec::new();
    }

    let mut possible: Vec<u8> = (1..=9u8).collect();

    // Retirer les valeurs de la ligne
    possible.retain(|&v| !grid.cells[row].contains(&v));

    // Retirer les valeurs de la colonne
    possible.retain(|&v| (0..9).all(|r| grid.cells[r][col] != v));

    // Retirer les valeurs du bloc
    let box_row = 3 * (row / 3);
    let box_col = 3 * (col / 3);
    possible.retain(|&v| {
        (box_row..box_row + 3).all(|r| (box_col..box_col + 3).all(|c| grid.cells[r][c] != v))
    });

    possible
}

#[allow(dead_code)]
fn find_mrv_empty(grid: &Grid) -> Option<(usize, usize, Vec<u8>)> {
    let mut best: Option<(usize, usize, Vec<u8>)> = None;
    let mut best_count = 10usize;

    for r in 0..9 {
        for c in 0..9 {
            if grid.cells[r][c] == 0 {
                let possible = get_possible_values(grid, r, c);
                if possible.len() < best_count {
                    best_count = possible.len();
                    best = Some((r, c, possible));
                    if best_count == 0 {
                        return best; // Echec immediat
                    }
                }
            }
        }
    }

    best
}

#[allow(unused_variables)]
fn count_solutions(grid: &Grid, max_solutions: usize) -> usize {
    // TODO etudiant : adaptez le backtracking pour compter toutes les solutions
    // au lieu de s'arreter a la premiere.
    // Indications :
    //   1. Copiez la grille avec grid.clone()
    //   2. Ecrivez une fonction recursive qui explore toutes les branches
    //   3. Quand la grille est complete, retournez 1 (une solution trouvee)
    //   4. Si nb_solutions >= max_solutions, arretez prematurement
    //   5. Apres chaque essai, remettez la case a 0 (backtrack)
    0 // TODO etudiant : remplacer par l'implementation
}

// Test : un bon puzzle devrait avoir exactement 1 solution
let grid_to_count = Grid::from_string(&easy_puzzles[0]);
let nb = count_solutions(&grid_to_count, 2);
println!("Nombre de solutions: {}", nb); // Doit afficher 1

Nombre de solutions: 0


### Test de validation sur puzzle mal formé

Pour vérifier que `count_solutions` explore bien toutes les branches (et ne s'arrête pas à la première solution), on le teste sur un **puzzle mal formé** : une grille très peu remplie qui admet **plusieurs solutions valides**.

**Pourquoi un puzzle mal formé ?**

- Un puzzle bien formé a une solution unique, donc `count_solutions` retournerait `1` dans tous les cas, ce qui ne prouve pas que l'exploration complète fonctionne.
- Un puzzle avec beaucoup de trous a de nombreuses solutions, donc on s'attend à `count_solutions >= 2`, ce qui valide la branche d'exploration complète au-delà de la première solution trouvée.

**Résultat attendu** : sur cette grille très lâche, `count_solutions` doit retourner **au moins 2** (et probablement plus, selon la valeur de `max_solutions`).

In [13]:
let sudoku = "
7.. .2. ...
23. ... 1..
..4 .8. ..2

... ... .5.
.6. 859 ..4
... ... .1.

5.6 .3. ...
... ... ...
... ... ...
";
let invalid_grid = Grid::from_string(sudoku);
let nb = count_solutions(&invalid_grid, 2);
println!("Nombre de solutions: {}", nb); // Doit afficher >= 2 (stub: 0 tant que non complete)

Nombre de solutions: 0


## Exercice : Nombre de placements valides pour la première colonne

### Énoncé

Etant donnée une grille de Sudoku partiellement remplie, écrivez une fonction `count_first_col_placements(grid)` qui compte le nombre de **façons valides et distinctes** de remplir toutes les cases vides de la **première colonne** (colonne d'indice 0).

Contrairement à `count_solutions` qui résout toute la grille, ici vous ne vous intéressez qu'à la première colonne. Pour chaque case vide de la colonne 0, essayez les valeurs possibles en respectant les contraintes du Sudoku, puis comptez le nombre total de combinaisons valides.

> **Indices :**

- Parcourez la première colonne case par case (ligne 0 à ligne 8)
- Pour chaque case vide, déterminez les valeurs possibles avec `is_valid_placement`
- Utilisez un backtracking qui ne porte que sur les 9 cases de la colonne 0
- Les cases déjà remplies dans la colonne 0 sont des contraintes fixées (à exclure des valeurs possibles)
- Le résultat est le nombre de combinaisons valides pour remplir toute la colonne ; la fonction rend `Option<usize>` et `None` tant qu'elle n'est pas complétée

In [14]:
// EXERCICE : Nombre de placements valides pour la premiere colonne

#[allow(unused_variables)]
fn count_first_col_placements(grid: &Grid) -> Option<usize> {
    // TODO etudiant : implementez cette fonction
    // 1. Identifiez les cases vides dans la colonne 0
    // 2. Pour chaque case vide, trouvez les valeurs possibles
    // 3. Utilisez un backtracking pour explorer toutes les combinaisons
    // 4. Comptez le nombre total de combinaisons valides
    None
}

// Test avec le puzzle facile
let test_grid_col = Grid::from_string(&easy_puzzles[0]);
let nb_col = count_first_col_placements(&test_grid_col);
println!("Nombre de placements valides pour la colonne 0: {:?}", nb_col);

// Test avec un puzzle plus contraint (moins de cases vides en colonne 0)
let grid_col2 = Grid::from_string(&hard_puzzles[0]);
let nb_col2 = count_first_col_placements(&grid_col2);
println!("Nombre de placements valides (puzzle difficile): {:?}", nb_col2);

Nombre de placements valides pour la colonne 0: None


Nombre de placements valides (puzzle difficile): None


## Synthèse et perspectives

Ce notebook a exploré l'algorithme de backtracking pour la résolution de Sudoku, depuis l'implémentation récursive naïve jusqu'à l'optimisation par l'heuristique MRV (Minimum Remaining Values). Les benchmarks ont montré une différence spectaculaire sur le nombre d'appels récursifs : le backtracking simple peut en nécessiter jusqu'à 335 000 sur les puzzles difficiles, tandis que MRV réduit ce nombre à quelques dizaines ou centaines — jusqu'à 2 621 fois moins d'appels. Le temps mur, lui, ne suit pas toujours : il baisse sur deux des cinq puzzles (10.22 → 0.43 ms et 11.73 → 0.20 ms) et monte sur les trois autres, chaque appel MRV balayant les cases vides coûtant plus cher qu'un appel simple. Le rendu texte a permis de distinguer clairement les valeurs initiales des valeurs déduites par le solveur.

L'heuristique MRV illustre le principe "fail-first" fondamental en recherche : en traitant d'abord les variables les plus contraintes, on détecte les impasses plus tôt et on élaguera massivement l'arbre de recherche. Cette leçon s'applique au-delà du Sudoku à tous les problèmes de satisfaction de contraintes. Les exercices proposés (comptage de solutions, placements valides sur une colonne) approfondissent la maîtrise du backtracking et de ses variantes.

Le carnet Python suivant, [Sudoku-02-DancingLinks-Python](Sudoku-02-DancingLinks-Python.ipynb), aborde une approche radicalement différente : la reformulation du Sudoku comme problème de couverture exacte, résolue par l'algorithme X de Knuth et la technique des Dancing Links (DLX).


## À retenir

Le backtracking est le solveur de référence : il essaie une valeur par case vide, rétroagit au premier conflit, et explore ainsi l'arbre des solutions. Sur la grille témoin (36 cases vides), il résout en **moins de 0,01 ms** (l'affichage arrondit à `0.00 ms`) avec **49 appels récursifs** ; sur les 11 puzzles « difficiles » du benchmark, **11/11 sont résolus** (temps moyen **4.57 ms**).

- Le backtracking naïf est simple, mais son nombre d'appels explose avec la difficulté : **24 776 appels** en moyenne sur les puzzles faciles, **95 455** sur les difficiles.
- **MRV** (Minimum Remaining Values) — choisir d'abord la case au plus petit domaine de valeurs restantes — réduit drastiquement l'exploration : **1334.6 fois moins d'appels** sur un puzzle difficile (228 215 → 171 appels, temps 10.22 → 0.43 ms).
- La récursivité rend le backtracking lisible : chaque appel tente une valeur, et l'appel parent « annule » la tentative au retour sur conflit.

## Vérifiez votre compréhension

**1. Combien d'appels récursifs le solveur simple effectue-t-il sur la grille témoin, et combien de cases vides cette grille compte-t-elle ?**

<details><summary>Réponse</summary>

La grille de test compte **36 cases vides** et le solveur simple l'exploite en **49 appels récursifs** pour la résoudre en **moins de 0,01 ms** (l'affichage arrondit à `0.00 ms`). C'est un puzzle « facile » : l'arbre d'exploration reste petit.

</details>

**2. Quel est l'effet mesuré de la heuristique MRV face au backtracking naïf ?**

<details><summary>Réponse</summary>

Sur un puzzle difficile, le backtracking simple effectue **228 215 appels**, alors que la variante MRV n'en effectue que **171** — un rapport de **1334.6** sur le nombre d'appels. Le temps mur, lui, ne suit pas mécaniquement ce rapport : il baisse sur deux des cinq puzzles et monte sur les trois autres. MRV choisit toujours la case dont le domaine de valeurs restantes est le plus petit, ce qui déclenche les échecs (et donc les rétroactions) le plus tôt possible.

</details>

**3. Pourquoi les « appels moyens » passent-ils de 24 776 (faciles) à 95 455 (difficiles) ?**

<details><summary>Réponse</summary>

C'est la signature de l'explosion combinatoire du backtracking : plus le puzzle est contraint, plus l'arbre à parcourir avant de trouver la solution grossit. Le benchmark mesure **24 776 appels moyens** sur les 10 puzzles faciles contre **95 455** sur les 11 puzzles difficiles.

</details>

## Pour aller plus loin

- **[Sudoku-02 — Dancing Links](Sudoku-02-DancingLinks-Python.ipynb)** : le carnet suivant — l'Algorithme X de Knuth et les listes liées dansantes, un solveur exact bien plus rapide.

- [Sudoku-03 — Algorithme Génétique](Sudoku-03-Genetic-Python.ipynb) : une approche par métaheuristique, en contraste avec l'exactitude du backtracking.

### Lectures

- Russell & Norvig, *Artificial Intelligence: A Modern Approach* — le backtracking et les heuristiques d'ordonnancement (approfondis dans le carnet [Sudoku-06](Sudoku-06-AIMA-CSP-Python.ipynb)).
